In [ ]:
# sweep_L8_riccati.py  (single-file driver)
betas  = [5.60 + 0.02*k for k in range(21)]
c0s    = [0.22, 0.24, 0.25, 0.26, 0.28]
R0s    = [0.035, 0.045, 0.050, 0.055, 0.065]
seeds  = list(range(16))

for c0 in c0s:
    for R0 in R0s:
        for beta in betas:
            stats = []
            for s in seeds:
                U = make_config(L=8, beta=beta, seed=s, protocol="right-inv")
                H  = pulled_wilson_hessian(U, L=8, beta=beta)          # your current
                Hs = riccati_stabilize(H, c0=c0, R0=R0)                 # your step
                lam_min = lanczos_min_eig(Hs, k=8)
                r_stab  = backtrack_stability_radius(U, Hs)             # monotone test
                gamma2  = curvature_proxy(Hs, c0)                        # your Γ2 proxy
                stats.append((lam_min, r_stab, gamma2))
            summarize_and_log(beta, c0, R0, stats)


NameError: name 'make_config' is not defined

In [ ]:
import numpy as np

# ============================
# 1) EDIT THESE ARRAYS
# ============================
# For each β_i where you have a lattice-unit mass gap m_lat(β_i),
# put your curvature scale μ(β_i) and the corresponding m_lat(β_i).
# Lengths must match; order must correspond.

beta = np.array([5.7, 5.8, 5.9, 6.0, 6.1])      # example β grid (edit)
mu   = np.array([0.92, 0.81, 0.74, 0.68, 0.63]) # your curvature scale μ(β) (edit)
mLat = np.array([0.88, 0.78, 0.71, 0.66, 0.61]) # lattice mass gap in lattice units (edit)

assert len(mu)==len(mLat)==len(beta)

# (Optional) if you also know the lattice spacing a(β) in lattice units (i.e., just 'a' itself),
# you can test μ(a) ~ a^{-p}. Otherwise, leave a=None.
a = None
# a = np.array([...])   # e.g., from scale setting; same length as beta

# ============================
# 2) PROPORTIONALITY FIT m_lat = k * μ
# ============================
# Constrained least squares with zero intercept -> k = <μ,m>/<μ,μ>
k = float(np.dot(mu, mLat) / np.dot(mu, mu))
pred = k * mu
res  = mLat - pred
rss  = float(np.dot(res, res))
tss  = float(np.dot(mLat - np.mean(mLat), mLat - np.mean(mLat)))
R2   = 1.0 - rss / tss if tss > 0 else np.nan

print("=== Proportionality test: m_lat = k * μ ===")
print(f"k (slope, no intercept) = {k:.6g}")
print(f"R^2                    = {R2:.6f}")
print(f"RMS residual           = {np.sqrt(rss/len(mu)):.6g}")
print("Residuals (m_lat - k*μ):", np.round(res, 6))

# ============================
# 3) OPTIONAL: POWER LAW μ(a) ~ A * a^{-p}
# ============================
if a is not None:
    assert len(a)==len(mu)
    mask = (a>0) & (mu>0)
    aa   = a[mask]
    mm   = mu[mask]
    x    = np.log(aa)
    y    = np.log(mm)
    # Linear fit: y = log A - p * x
    X    = np.vstack([np.ones_like(x), -x]).T
    theta, *_ = np.linalg.lstsq(X, y, rcond=None)
    logA, p = theta[0], theta[1]
    yhat = X @ theta
    R2p  = 1.0 - np.sum((y - yhat)**2)/np.sum((y - np.mean(y))**2)

    print("\n=== Scaling test: μ(a) ~ A * a^{-p} ===")
    print(f"p (exponent) = {p:.6g}")
    print(f"A            = {np.exp(logA):.6g}")
    print(f"R^2 (log fit)= {R2p:.6f}")


=== Proportionality test: m_lat = k * μ ===
k (slope, no intercept) = 0.962363
R^2                    = 0.998237
RMS residual           = 0.00396962
Residuals (m_lat - k*μ): [-0.005374  0.000486 -0.002149  0.005593  0.003711]


In [ ]:
# ri_tangent_kernel.py
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")

import jax
import jax.numpy as jnp
from jax import jit, vmap
from functools import partial

jax.config.update("jax_enable_x64", True)

# =========================
# su(3) basis (Hermitian)
# =========================
def gell_mann():
    e11 = jnp.array([[1,0,0],[0,0,0],[0,0,0]], dtype=jnp.complex128)
    e22 = jnp.array([[0,0,0],[0,1,0],[0,0,0]], dtype=jnp.complex128)
    e33 = jnp.array([[0,0,0],[0,0,0],[0,0,1]], dtype=jnp.complex128)

    l1 = jnp.array([[0,1,0],[1,0,0],[0,0,0]], dtype=jnp.complex128)
    l2 = jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]], dtype=jnp.complex128)
    l3 = e11 - e22
    l4 = jnp.array([[0,0,1],[0,0,0],[1,0,0]], dtype=jnp.complex128)
    l5 = jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]], dtype=jnp.complex128)
    l6 = jnp.array([[0,0,0],[0,0,1],[0,1,0]], dtype=jnp.complex128)
    l7 = jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]], dtype=jnp.complex128)
    l8 = (e11 + e22 - 2*e33) / jnp.sqrt(3.0)
    L = jnp.stack([l1,l2,l3,l4,l5,l6,l7,l8], axis=0)  # (8,3,3) Hermitian
    return L

GM = gell_mann()

# Hermitian H -> anti-Hermitian X with Tr(X)=0
@jit
def herm_to_anti(H):
    return 1j * H

# Flatten/unflatten 8-dim coefficients ↔ su(3) matrix
@jit
def coeffs_to_X(c8):
    # c8 real coefficients: H = sum c_a * lambda_a  (Hermitian)
    H = jnp.tensordot(c8, GM, axes=1)  # (3,3)
    return herm_to_anti(H)             # anti-Hermitian

@jit
def X_to_coeffs(X):
    # project X (anti-Hermitian) back to Hermitian via -i X
    H = -1j * X                        # Hermitian
    GM_flat = GM.reshape(8, 9)         # (8,9)
    H_flat = H.reshape(9,)             # (9,)

    # num_a = Re Tr(lambda_a H)
    num = jnp.real(GM_flat.conj() @ H_flat)  # (8,)

    # den_a = Re Tr(lambda_a^2) for each a
    den = jnp.real(jnp.sum(GM_flat.conj() * GM_flat, axis=1))  # (8,)

    return num / den                   # (8,)

# Right-invariant pushforward δU = U X
@jit
def tangent_push(U, c8):
    X = coeffs_to_X(c8)
    return U @ X  # (3,3) anti-Hermitian in the fiber at U

# Small-step exp map for probes: U' = U exp(ε X)
@jit
def exp_UX(U, c8, eps):
    X = coeffs_to_X(c8)
    # 3x3 matrix exp via eig; stable enough for small eps.
    w, V = jnp.linalg.eig(X)
    expX = V @ jnp.diag(jnp.exp(w)) @ jnp.linalg.inv(V)
    return (U @ expX).astype(jnp.complex128)

# =========================
# Lattice batching helpers
# =========================
# U: (..., 3,3); c: (..., 8)
@partial(jit, static_argnames=("axis_mat","axis_coeff"))
def batch_tangent_push(U, c, axis_mat=-3, axis_coeff=-2):
    f = vmap(tangent_push, in_axes=(0,0))
    Um = U.reshape((-1,3,3))
    Cm = c.reshape((-1,8))
    dU = f(Um, Cm).reshape(U.shape)
    return dU

@partial(jit, static_argnames=("axis_mat","axis_coeff"))
def batch_exp_UX(U, c, eps, axis_mat=-3, axis_coeff=-2):
    f = vmap(exp_UX, in_axes=(0,0,None))
    Um = U.reshape((-1,3,3))
    Cm = c.reshape((-1,8))
    Up = f(Um, Cm, eps).reshape(U.shape)
    return Up

# =========================
# Public kernel API
# =========================
class RITangent:
    """
    Standalone right-invariant tangent kernel.
    - All ops are pure, JIT-able, and batched.
    - No dependency on action/HOTRG; pass P projector if needed upstream.
    """
    def __init__(self):
        self.GM = GM

    @staticmethod
    @jit
    def to_mat(c8):           # R^8 -> su(3) (anti-Hermitian)
        return coeffs_to_X(c8)

    @staticmethod
    @jit
    def to_coeffs(X):         # su(3) -> R^8
        return X_to_coeffs(X)

    @staticmethod
    @jit
    def push(U, c8):          # δU = U X(c8)
        return tangent_push(U, c8)

    @staticmethod
    @jit
    def step(U, c8, eps):     # U' = U exp(ε X(c8))
        return exp_UX(U, c8, eps)

    @staticmethod
    def push_batched(U, C):   # shapes: U[...,3,3], C[...,8]
        return batch_tangent_push(U, C)

    @staticmethod
    def step_batched(U, C, eps):
        return batch_exp_UX(U, C, eps)

# ========== minimal self-test ==========
if __name__ == "__main__":
    key = jax.random.PRNGKey(0)
    # random 3x3 complex, QR -> U(3), then normalize det to 1
    Z = jax.random.normal(key, (3,3)) + 1j*jax.random.normal(key, (3,3))
    Q, R = jnp.linalg.qr(Z)

    detQ = jnp.linalg.det(Q)
    U = Q / detQ ** (1.0/3.0)

    c8 = jnp.arange(8.0)

    dU = RITangent.push(U, c8)
    U2 = RITangent.step(U, c8, 1e-3)

    # roundtrip coeffs
    X = RITangent.to_mat(c8)
    c_rt = RITangent.to_coeffs(X)

    print("‖c - c_rt‖:", float(jnp.linalg.norm(c8 - c_rt)))
    print("smoke:",
          jnp.isfinite(jnp.linalg.norm(dU)).item(),
          jnp.isfinite(jnp.linalg.norm(U2)).item())


‖c - c_rt‖: 0.0
smoke: True True


In [ ]:
# ============================================================
# MINIMAL SU(3) CONFIG GENERATOR (Protocol B – Right-Invariant)
# ============================================================

import jax
import jax.numpy as jnp
from jax import random

def _project_to_su3(M):
    # Make anti-Hermitian traceless
    A = (M - M.conj().T) / 2
    A = A - jnp.trace(A)/3 * jnp.eye(3, dtype=A.dtype)
    return A

def _exp_su3(A):
    # 3×3 matrix exponential
    return jax.scipy.linalg.expm(A)

def random_su3_matrix(key, scale=0.3):
    # Draw a random tangent vector and exponentiate
    M = random.normal(key, (3,3), dtype=jnp.float64) + 1j*random.normal(key, (3,3), dtype=jnp.float64)
    A = _project_to_su3(M)
    return _exp_su3(scale * A)

def make_config(L, beta, seed, protocol="right-inv"):
    # Returns a JAX array of shape (L,L,L,L,4,3,3)
    key = random.PRNGKey(seed)

    # Wilson β is not used directly here — you thermalize later in your pipeline.
    # This simply produces a valid random SU(3) field.

    keys = random.split(key, L*L*L*L*4)
    mats = jnp.stack([random_su3_matrix(k) for k in keys], axis=0)
    U = mats.reshape(L, L, L, L, 4, 3, 3)

    return U


In [ ]:
# Colab-ready JAX script for SU(2) Wilson-action Hessian on an A100
# No jit tricks, so you avoid the "L is a tracer" reshape error.

import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.lib import xla_bridge

jax.config.update("jax_enable_x64", True)

# ---------- SU(2) helpers ----------

def pauli_matrices():
    """Return Pauli matrices as a (3,2,2) complex128 array."""
    sigma_x = jnp.array([[0, 1], [1, 0]], dtype=jnp.complex128)
    sigma_y = jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex128)
    sigma_z = jnp.array([[1, 0], [0, -1]], dtype=jnp.complex128)
    return jnp.stack([sigma_x, sigma_y, sigma_z], axis=0)

SIGMA = pauli_matrices()

def su2_from_vec(a):
    """
    Map a 3-vector a \in R^3 to SU(2) via exp(i/2 * a·sigma).
    a: shape (3,)
    Returns: shape (2,2) complex128
    """
    a = jnp.asarray(a, dtype=jnp.float64)
    theta = jnp.linalg.norm(a)
    half = 0.5 * theta

    # sinc(x) = sin(x)/x with safe small-theta branch
    sinc = jnp.where(
        theta < 1e-12,
        0.5 - (theta**2) / 48.0,  # series for sin(theta/2)/theta
        jnp.sin(half) / theta
    )

    dot = jnp.tensordot(a, SIGMA, axes=1)  # sum_a a_a sigma_a
    U = jnp.cos(half) * jnp.eye(2, dtype=jnp.complex128) + 1j * sinc * dot
    return U

# ---------- Lattice geometry helpers ----------

def shift_site(x, y, z, t, direction, L):
    """
    Periodic shift by +1 in coordinate 'direction' (0=x,1=y,2=z,3=t).
    """
    if direction == 0:
        return ( (x + 1) % L, y, z, t )
    if direction == 1:
        return ( x, (y + 1) % L, z, t )
    if direction == 2:
        return ( x, y, (z + 1) % L, t )
    if direction == 3:
        return ( x, y, z, (t + 1) % L )
    raise ValueError("direction must be 0..3")

# ---------- Wilson action ----------

def wilson_action_su2_flat(link_vars_flat, beta, L):
    """
    SU(2) Wilson action on an L^4 lattice, given link variables in R^3 coords.

    link_vars_flat: shape (L^4 * 4 * 3,) real (algebra coords)
                    ordering: [x,y,z,t,mu,3]
    beta: Wilson coupling
    L: lattice extent
    """
    # reshape to (x,y,z,t,mu,3)
    links = link_vars_flat.reshape(L, L, L, L, 4, 3)

    total_S = 0.0
    for x in range(L):
        for y in range(L):
            for z in range(L):
                for t in range(L):
                    for mu in range(4):
                        for nu in range(mu + 1, 4):
                            # U_mu(x)
                            U1 = su2_from_vec(links[x, y, z, t, mu])

                            # U_nu(x + mu)
                            xm = shift_site(x, y, z, t, mu, L)
                            U2 = su2_from_vec(links[xm[0], xm[1], xm[2], xm[3], nu])

                            # U_mu(x + nu)
                            xn = shift_site(x, y, z, t, nu, L)
                            U3 = su2_from_vec(links[xn[0], xn[1], xn[2], xn[3], mu])

                            # U_nu(x)
                            U4 = su2_from_vec(links[x, y, z, t, nu])

                            # Plaquette U_p = U_mu(x) U_nu(x+mu) U_mu^\dagger(x+nu) U_nu^\dagger(x)
                            Up = U1 @ U2 @ jnp.conjugate(U3.T) @ jnp.conjugate(U4.T)

                            # SU(2) Wilson plaquette term: S_p = (beta/2) * (2 - Re Tr U_p)
                            S_p = (beta / 2.0) * (2.0 - jnp.real(jnp.trace(Up)))
                            total_S = total_S + S_p

    return total_S

# ---------- Hessian + spectrum ----------

def compute_hessian_spectrum_su2(beta=2.0, L=2):
    """
    Compute full Hessian of SU(2) Wilson action at vacuum and its eigenvalues.

    Returns:
        eigs: jnp.ndarray of eigenvalues (sorted ascending)
    """
    dof = (L**4) * 4 * 3  # 4 directions, 3 algebra comps
    x0 = jnp.zeros((dof,), dtype=jnp.float64)  # vacuum: all links = identity

    print(f"[SU(2)] L={L}, beta={beta}, DOF={dof}")
    print("Computing Hessian (this may take a bit on first run)...")

    t0 = time.time()
    # No jit here: avoids all static_argnums / tracer-shape issues.
    H_fn = jax.hessian(lambda x: wilson_action_su2_flat(x, beta, L))
    H = H_fn(x0)
    t1 = time.time()

    print(f"Hessian shape: {H.shape}, computed in {t1 - t0:.2f} s")

    eigs = jnp.linalg.eigvalsh(H)
    print("Min eigenvalue:", float(eigs[0]))
    print("Max eigenvalue:", float(eigs[-1]))
    return eigs

# ---------- Main entry ----------

if __name__ == "__main__":
    print("JAX backend platform:", xla_bridge.get_backend().platform)
    eigs = compute_hessian_spectrum_su2(beta=2.0, L=2)

    # Print a few eigenvalues as sanity check
    eigs_np = np.array(eigs)
    print("First 10 eigenvalues:", eigs_np[:10])
    print("Last 10 eigenvalues:", eigs_np[-10:])


<>:25: SyntaxWarning: invalid escape sequence '\i'
<>:25: SyntaxWarning: invalid escape sequence '\i'
/tmp/ipython-input-2457176234.py:25: SyntaxWarning: invalid escape sequence '\i'
  Map a 3-vector a \in R^3 to SU(2) via exp(i/2 * a·sigma).
/tmp/ipython-input-2457176234.py:135: DeprecationWarning: jax.lib.xla_bridge.get_backend is deprecated and will be removed in JAX v0.8.0; use jax.extend.backend.get_backend.
  print("JAX backend platform:", xla_bridge.get_backend().platform)


JAX backend platform: gpu
[SU(2)] L=2, beta=2.0, DOF=192
Computing Hessian (this may take a bit on first run)...


/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:5473: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


Hessian shape: (192, 192), computed in 52.43 s
Min eigenvalue: nan
Max eigenvalue: nan
First 10 eigenvalues: [nan nan nan nan nan nan nan nan nan nan]
Last 10 eigenvalues: [nan nan nan nan nan nan nan nan nan nan]


In [5]:
# SU(2) Wilson Hessian on 2^4 lattice, Hessian at vacuum (no NaNs version)
import jax
import jax.numpy as jnp
from jax.lib import xla_bridge

jax.config.update("jax_enable_x64", True)

print("JAX backend platform:", xla_bridge.get_backend().platform)

# ---------------- SU(2) algebra ----------------

def pauli_matrices():
    """Return the 3 Pauli matrices as (3,2,2) complex128 array."""
    sigma = [
        jnp.array([[0, 1], [1, 0]], dtype=jnp.complex128),
        jnp.array([[0, -1j], [1j, 0]], dtype=jnp.complex128),
        jnp.array([[1, 0], [0, -1]], dtype=jnp.complex128),
    ]
    return jnp.stack(sigma, axis=0)

SIGMA = pauli_matrices()

def su2_exp_taylor(a_vec):
    r"""
    Second–order Taylor for exp(i/2 * a·σ).

    This is **exact up to quadratic order in a**, which is all we need
    for the Hessian at a = 0. No trig, no divisions, no 0/0.
    """
    # a_vec shape: (3,)
    # X = (1/2) a·σ
    X = 0.5 * jnp.tensordot(a_vec, SIGMA, axes=1)  # (2,2) complex
    I = jnp.eye(2, dtype=jnp.complex128)
    # exp(i X) ≈ I + i X - (X^2)/2  + O(||a||^3)
    return I + 1j * X - 0.5 * (X @ X)


# ---------------- Lattice plumbing ----------------

def wilson_action_single_plaquette(U1, U2, U3, U4, beta):
    """
    One SU(2) plaquette contribution:
        S_p = (beta/2) * (2 - Re Tr(U_p)),
    with U_p = U1 U2 U3^\dagger U4^\dagger.
    """
    Up = U1 @ U2 @ jnp.conjugate(U3.T) @ jnp.conjugate(U4.T)
    tr = jnp.trace(Up)
    return (beta / 2.0) * (2.0 - jnp.real(tr))


def lattice_wilson_action(link_vars_flat, L=2, beta=2.0):
    """
    Wilson action on an L^4 SU(2) lattice.

    link_vars_flat: shape (L^4 * 4 * 3,) real
        For each site (x,y,z,t), direction mu, and color a=1..3,
        we store an su(2) algebra vector a_vec[mu,a].
    """
    # Reshape to (x,y,z,t,mu,3)
    links = link_vars_flat.reshape(L, L, L, L, 4, 3)

    S_total = 0.0

    # Naive explicit loops – fine for L=2 and keeps things readable.
    for x in range(L):
        for y in range(L):
            for z in range(L):
                for t in range(L):
                    loc = (x, y, z, t)

                    for mu in range(4):
                        for nu in range(mu + 1, 4):
                            # x
                            x0, y0, z0, t0 = loc

                            # x + μ
                            x_mu = (x0 + (mu == 0)) % L
                            y_mu = (y0 + (mu == 1)) % L
                            z_mu = (z0 + (mu == 2)) % L
                            t_mu = (t0 + (mu == 3)) % L

                            # x + ν
                            x_nu = (x0 + (nu == 0)) % L
                            y_nu = (y0 + (nu == 1)) % L
                            z_nu = (z0 + (nu == 2)) % L
                            t_nu = (t0 + (nu == 3)) % L

                            # U1 at x, mu
                            a1 = links[x0, y0, z0, t0, mu]
                            U1 = su2_exp_taylor(a1)

                            # U2 at x+μ, nu
                            a2 = links[x_mu, y_mu, z_mu, t_mu, nu]
                            U2 = su2_exp_taylor(a2)

                            # U3 at x+ν, mu
                            a3 = links[x_nu, y_nu, z_nu, t_nu, mu]
                            U3 = su2_exp_taylor(a3)

                            # U4 at x, nu
                            a4 = links[x0, y0, z0, t0, nu]
                            U4 = su2_exp_taylor(a4)

                            S_total = S_total + wilson_action_single_plaquette(
                                U1, U2, U3, U4, beta
                            )

    # S_total is real (float64) scalar
    return S_total


# ---------------- Hessian computation ----------------

def compute_hessian_spectrum(beta=2.0, L=2):
    """
    Compute the full Hessian of S_W at the vacuum (all links = identity),
    i.e. link_vars_flat = 0, and return its eigenvalues.
    """
    # Number of real DOF: sites * directions * su(2) generators
    N_links = (L ** 4) * 4 * 3
    x0 = jnp.zeros(N_links, dtype=jnp.float64)

    print(f"[SU(2)] L={L}, beta={beta}, DOF={N_links}")

    # H_ij = ∂² S / ∂x_i ∂x_j at x=0
    # No jit: easier to debug; for N=192 this is still fine.
    hessian_fn = jax.jacfwd(jax.jacrev(lattice_wilson_action))
    H = hessian_fn(x0, L=L, beta=beta)

    # Symmetrize numerically just in case of tiny AD asymmetries
    H = 0.5 * (H + H.T)

    eigs = jnp.linalg.eigvalsh(H)
    eigs = jnp.sort(eigs)

    print("Hessian shape:", H.shape)
    print("Min eigenvalue:", float(eigs[0]))
    print("Max eigenvalue:", float(eigs[-1]))
    print("First 10 eigenvalues:", eigs[:10])
    print("Last 10 eigenvalues:", eigs[-10:])

    return eigs

# ---- run it ----
if __name__ == "__main__":
    eigs = compute_hessian_spectrum(beta=2.0, L=2)


<>:44: SyntaxWarning: invalid escape sequence '\d'
<>:44: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-2107874415.py:44: SyntaxWarning: invalid escape sequence '\d'
  with U_p = U1 U2 U3^\dagger U4^\dagger.


JAX backend platform: cpu
[SU(2)] L=2, beta=2.0, DOF=192


/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:2803: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


Hessian shape: (192, 192)
Min eigenvalue: -7.862020267044665e-15
Max eigenvalue: 8.000000000000002
First 10 eigenvalues: [-7.86202027e-15 -4.49374990e-15 -4.04364902e-15 -3.89165639e-15
 -3.85816329e-15 -3.84037888e-15 -3.47162408e-15 -2.99387145e-15
 -2.19621401e-15 -1.81777150e-15]
Last 10 eigenvalues: [6. 8. 8. 8. 8. 8. 8. 8. 8. 8.]
